In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install transformers wandb -q

# Model 2 (pretrained BERT)

The idea is to format each question+option pair as:
* [CLS] question [SEP] option [SEP]
* and fine-tune BERT to classify which option is correct.

In [3]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception as e:
    print(f"Could not load WANDB_API_KEY from Kaggle Secrets ({e}). "
          f"Falling back to offline wandb logging so the rest of the notebook still runs.")
    os.environ["WANDB_MODE"] = "offline"

# ---- Load Data ----
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# ---- Config ----
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

wandb.init(project="24f3002284-t22026", name="model2-bert", config={
    "model": MODEL_NAME,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "lr": LR
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
options = ['A', 'B', 'C', 'D', 'E']


def map_at_3(y_true_idx, logits):
    """MAP@3 -- the competition's actual leaderboard metric."""
    top3 = torch.topk(logits, 3, dim=1).indices
    scores = []
    for true_idx, pred_idx in zip(y_true_idx, top3):
        pred_list = pred_idx.tolist()
        scores.append(1.0 / (pred_list.index(true_idx) + 1) if true_idx in pred_list else 0.0)
    return sum(scores) / len(scores)


# Dataset 
class MCQDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.df = df.reset_index(drop=True)
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encodings = []
        for opt in options:
            enc = tokenizer(
                str(row['prompt']),
                str(row[opt]),
                max_length=MAX_LEN,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            encodings.append({
                'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze()
            })
        if not self.is_test:
            label = torch.tensor(label_map[row['answer']], dtype=torch.long)
            return encodings, label
        return encodings


# Model
class BERTMCQModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, encodings):
        logits = []
        for enc in encodings:
            output = self.bert(
                input_ids=enc['input_ids'],
                attention_mask=enc['attention_mask']
            )
            cls = output.last_hidden_state[:, 0, :]
            logits.append(self.classifier(cls))
        return torch.cat(logits, dim=1)  # (batch, 5)


# Collate
def collate_fn(batch):
    if isinstance(batch[0], tuple):
        encodings_list, labels = zip(*batch)
        labels = torch.stack(labels)
    else:
        encodings_list = batch
        labels = None

    batch_encodings = []
    for i in range(len(options)):
        input_ids = torch.stack([e[i]['input_ids'] for e in encodings_list])
        attention_mask = torch.stack([e[i]['attention_mask'] for e in encodings_list])
        batch_encodings.append({'input_ids': input_ids, 'attention_mask': attention_mask})
    return batch_encodings, labels


# Training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Stratified split -- matches Model 1 & Model 3's splitting strategy so the three WandB runs are actually comparable, not just superficially similar.
train_df, val_df = train_test_split(
    train, test_size=0.2, random_state=42, stratify=train['answer']
)
train_loader = DataLoader(MCQDataset(train_df), batch_size=BATCH_SIZE,
                           shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(MCQDataset(val_df), batch_size=BATCH_SIZE,
                         collate_fn=collate_fn)

model = BERTMCQModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

best_val_f1 = -1
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    train_true, train_pred = [], []
    for encodings, labels in train_loader:
        encodings = [{k: v.to(device) for k, v in e.items()} for e in encodings]
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(encodings)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        train_true += labels.cpu().tolist()
        train_pred += logits.argmax(1).cpu().tolist()

    train_acc = accuracy_score(train_true, train_pred)
    train_f1 = f1_score(train_true, train_pred, average='macro')

    model.eval()
    val_true, val_pred, val_logits_all = [], [], []
    with torch.no_grad():
        for encodings, labels in val_loader:
            encodings = [{k: v.to(device) for k, v in e.items()} for e in encodings]
            labels = labels.to(device)
            logits = model(encodings)
            val_true += labels.cpu().tolist()
            val_pred += logits.argmax(1).cpu().tolist()
            val_logits_all.append(logits.cpu())

    val_acc = accuracy_score(val_true, val_pred)
    val_f1 = f1_score(val_true, val_pred, average='macro')
    val_map3 = map_at_3(val_true, torch.cat(val_logits_all, dim=0))

    print(f"Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, "
          f"Train Acc={train_acc:.4f}, Train F1={train_f1:.4f}, "
          f"Val Acc={val_acc:.4f}, Val F1={val_f1:.4f}, Val MAP@3={val_map3:.4f}")

    wandb.log({
        "epoch": epoch + 1, "loss": total_loss / len(train_loader),
        "train_acc": train_acc, "train_f1": train_f1,
        "val_acc": val_acc, "val_f1": val_f1, "val_map3": val_map3,
    })

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'model2_bert_best.pth')
        wandb.save('model2_bert_best.pth')

wandb.summary["best_val_f1"] = best_val_f1
wandb.finish()
print("Training done!")

# Inference -- loads the BEST checkpoint (by val F1), not whatever's left in memory from the last epoch.
model.load_state_dict(torch.load('model2_bert_best.pth', map_location=device))
model.eval()
test_dataset = MCQDataset(test, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

all_preds = []
with torch.no_grad():
    for encodings, _ in test_loader:
        encodings = [{k: v.to(device) for k, v in e.items()} for e in encodings]
        logits = model(encodings)
        top3 = torch.topk(logits, 3, dim=1).indices.cpu().numpy()
        all_preds.extend(top3)

predictions = [' '.join([options[i] for i in pred]) for pred in all_preds]
submission = pd.DataFrame({'ID': test['id'], 'Prediction': predictions})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print("Done!")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 24f3002284 (24f3002284-dl-genai-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260818_030137-qpmqsj41
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model2-bert
wandb: ⭐️ View project at https://wandb.ai/24f3002284-dl-genai-project/24f3002284-t22026
wandb: 🚀 View run at https://wandb.ai/24f3002284-dl-genai-project/24f3002284-t22026/runs/qpmqsj41


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch 1: Loss=1.1825, Train Acc=0.5194, Train F1=0.5148, Val Acc=0.9225, Val F1=0.9210, Val MAP@3=0.9587


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 2: Loss=0.3425, Train Acc=0.8775, Train F1=0.8732, Val Acc=0.9925, Val F1=0.9920, Val MAP@3=0.9958
Epoch 3: Loss=0.1238, Train Acc=0.9587, Train F1=0.9582, Val Acc=0.9950, Val F1=0.9949, Val MAP@3=0.9975


wandb: updating run metadata
wandb: uploading model2_bert_best.pth; uploading output.log; uploading wandb-summary.json
wandb: uploading model2_bert_best.pth
wandb: 
wandb: Run history:
wandb:     epoch ▁▅█
wandb:      loss █▂▁
wandb: train_acc ▁▇█
wandb:  train_f1 ▁▇█
wandb:   val_acc ▁██
wandb:    val_f1 ▁██
wandb:  val_map3 ▁██
wandb: 
wandb: Run summary:
wandb: best_val_f1 0.99493
wandb:       epoch 3
wandb:        loss 0.12375
wandb:   train_acc 0.95875
wandb:    train_f1 0.95816
wandb:     val_acc 0.995
wandb:      val_f1 0.99493
wandb:    val_map3 0.9975
wandb: 
wandb: 🚀 View run model2-bert at: https://wandb.ai/24f3002284-dl-genai-project/24f3002284-t22026/runs/qpmqsj41
wandb: ⭐️ View project at: https://wandb.ai/24f3002284-dl-genai-project/24f3002284-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 1 other file(s)
wandb: Find logs at: ./wandb/run-20260818_030137-qpmqsj41/logs


Training done!
   ID Prediction
0   1      A E D
1   2      B C A
2   3      B E D
3   4      E C D
4   5      C A D
Done!


In [4]:
# Inference with BERT Model
model.eval()
test_dataset = MCQDataset(test, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

all_preds = []
with torch.no_grad():
    for encodings, _ in test_loader:
        encodings = [{k: v.to(device) for k, v in e.items()} for e in encodings]
        logits = model(encodings)
        top3 = torch.topk(logits, 3, dim=1).indices.cpu().numpy()
        all_preds.extend(top3)

options = ['A', 'B', 'C', 'D', 'E']
predictions = [' '.join([options[i] for i in pred]) for pred in all_preds]

submission = pd.DataFrame({'ID': test['id'], 'Prediction': predictions})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print("Done!")

   ID Prediction
0   1      A E D
1   2      B C A
2   3      B E D
3   4      E C D
4   5      C A D
Done!
